In [ ]:
import pandas as pd

df = pd.read_csv("../data/creditsense_applications.csv")

print("Original dataset shape:", df.shape)

Importing the original dataset and ensurin it is the expected shape.

In [ ]:
df_model = df.dropna(
    subset=["savings_balance_gbp", "years_at_current_address"]
).copy()

print("Modelling dataset shape:", df_model.shape)
print("Rows excluded:", len(df) - len(df_model))

Removing the known missing data and confirming only those have been removed.

In [ ]:
excluded_features = [
    "application_id",
    "default_within_24m",
    "age_years",
    "sex",
    "marital_status",
    "residency_status",
    "num_dependents",
    "region",
    "loan_purpose",
    "application_channel",
    "previous_application_declined"
]

Duplicating the `excluded_features` list used in my baseline model.

In [ ]:
X = df_model.drop(columns=excluded_features)

y = df_model["default_within_24m"]

print("predictive features:", X.shape[1])
print("Target:", y.name)
print("\nSelected features:")
print(X.columns.tolist())

Duplicating the predictive features list and ensuring it has been successful.

In [ ]:
nominal_features = [
    "employment_status",
    "housing_status",
    "credit_history",
    "checking_account_status"
]

binary_features = [
    "has_savings_account",
    "has_guarantor"
]

numerical_features = [
    col for col in X.columns
    if col not in nominal_features + binary_features
]

Recreating the original feature split in order to define how I handle each.

In [ ]:
print("Nominal:", nominal_features)
print("Binary:", binary_features)
print("Numerical:", numerical_features)

print("\nTotal features:",
      len(nominal_features)
      + len(binary_features)
      + len(numerical_features))

Confirming the split has been successful.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Ran exactly the same split as in my baseline logistic regression model.

In [ ]:
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining default rate:")
print(y_train.value_counts(normalize=True))

print("\nTesting default rate:")
print(y_test.value_counts(normalize=True))

Ensuring the split has been successful.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

Imported all 3 transformers ready for comparison.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_features),
        ("bin", OrdinalEncoder(
            categories=[["No", "Yes"], ["No", "Yes"]]
        ), binary_features)
    ]
)

Set up for the preprocessing. This has not seen any data yet, it is simply to define each of the transformations in advance.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

Import the pipeline and model type.

In [ ]:
baseline_model = Pipeline(
    steps =[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

Created baseline model, but have not fitted it to any data yet.

In [ ]:
baseline_model.fit(X_train, y_train)

Fitted model to training data only, the preprocessing would have allowed the model to learn fromthe training data even at this stage. This allows the model to learn the weights on the training data. Having fitted on the training data only I prevent leakage from the held-out test set during fitting.

In [ ]:
coefficients = baseline_model.named_steps["classifier"].coef_

print(coefficients)
print("coefficient shape:", coefficients.shape)

Retrieving the weights from the model during training.

In [ ]:
feature_names = baseline_model.named_steps["preprocessor"].get_feature_names_out()

print(feature_names)
print("Numberof features:", len(feature_names))

Retrieveing the names of the post processed features.

In [ ]:
coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients.flatten()
})

coefficient_df

Paired column name with model coefficient. See Explainability notes in README.md for initial analysis of this information.

In [ ]:
example_applicant = X_train.iloc[[0]]

example_applicant

Viewing one applicants data in order to determine the information received by the model before preprocessing.

In [ ]:
transformed_applicant = baseline_model.named_steps["preprocessor"].transform(example_applicant)

print(transformed_applicant)
print(transformed_applicant.shape)

Checking transformed values using the previously learned parameters.

In [ ]:
applicant_values_df = pd.DataFrame({
    "Feature": feature_names,
    "Transformed Value": transformed_applicant.flatten(),
    "Coefficient": coefficients.flatten()
})

applicant_values_df

Created a comparable DataFrame for analysis.

In [ ]:
applicant_values_df["Contribution"] = (
    applicant_values_df["Transformed Value"] *
    applicant_values_df["Coefficient"]
)

applicant_values_df

Being able to see the Contribution enables me to determine the way the model treats different encoding methods. See the explainability section in my README.md file for further analysis.

In [ ]:
reference_categories = X_train[nominal_features].mode()

reference_categories

Confirming the categories we have used one-hot encoding on and what the most common value in each is ready to implement reference points. Refer to decision 013 in decision journal for further details.

In [ ]:
reference_categories = reference_categories.iloc[0].to_dict()

reference_categories

Turning the new reference points into a dictionary.

In [ ]:
applicant_values_df["Reference Value"] = 0.0

applicant_values_df

Setting all reference values to zero. This only changes the way the nominal categories will work in the long term.

In [ ]:
for feature, category in reference_categories.items():
    reference_feature = f"nom__{feature}_{category}"

    applicant_values_df.loc[
        applicant_values_df["Feature"] == reference_feature,
        "Reference Value"
    ] = 1.0

applicant_values_df

Changing the decided four rows reference value to 1.0.

In [ ]:
applicant_values_df["Adjusted Contribution"] = (
    (applicant_values_df["Transformed Value"] -
     applicant_values_df["Reference Value"]) *
     applicant_values_df["Coefficient"]
)

applicant_values_df

We can see from the above that the nominal values are the only one which have changed in allignment with our expectations.

In [ ]:
applicant_values_df["Original Feature"] = (
    applicant_values_df["Feature"].str.split("__", n=1).str[1]
)

for feature in nominal_features:
    applicant_values_df.loc[
        applicant_values_df["Feature"].str.startswith(f"nom__{feature}_"),
        "Original Feature"
    ] = feature

applicant_values_df

Labelling the rows so that we can group them together.

In [ ]:
grouped_contributions = applicant_values_df.groupby(
    "Original Feature"
)["Adjusted Contribution"].sum().reset_index()

grouped_contributions

Aggregating the adjusted contributions together ready for proof of concept.

In [ ]:
intercept = baseline_model.named_steps["classifier"].intercept_[0]

print(intercept)

Retrieving an intercept value.

In [ ]:
reference_adjustment = (
    applicant_values_df["Reference Value"] *
    applicant_values_df["Coefficient"]
).sum()

print(reference_adjustment)

Calculating the adjustment required with just the nominal reference categories.

In [ ]:
adjusted_baseline = intercept + reference_adjustment

print(adjusted_baseline)


Calculating adjusted starting score to reflect the changes to referemce points outlined in decision 013.

In [ ]:
reconstructed_log_odds = (
    adjusted_baseline +
    grouped_contributions["Adjusted Contribution"].sum()
)

print(reconstructed_log_odds)

Calculated reconstructed log-odds score.

In [ ]:
import numpy as np

reconstructed_probability = 1 / (1 + np.exp(-reconstructed_log_odds))

model_probability = baseline_model.predict_proba(example_applicant)[0, 1]

print("Reconstructed probability:", reconstructed_probability)
print("Actual model probability:", model_probability)
print("Values match:", np.isclose(reconstructed_probability, model_probability))

Verified that the adjusted reference points preserve the model's prediction, grouping the features down to 16 again preserved the total contribution and that the individual explanation faithfully reconstructed this applicants predicted probability.